In [ ]:
# I count connected components of kNN graph instead of runnning a ripser on it

In [ ]:
import sys
!{sys.executable} -m pip install dill --user

In [ ]:
# Monte Carlo k-NN Optimization for Topological Data Analysis
# Run this in Jupyter Notebook

import os
import dill
import numpy as np
import random
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.csgraph import connected_components
import matplotlib.pyplot as plt
from collections import Counter
import pickle
from tqdm import tqdm

class MonteCarloKNNOptimizer:
    def __init__(self, data_root='${TDL_ROOT_DIR}/results/D1', true_b0=9):
        self.data_root = data_root
        self.true_b0 = true_b0
        self.train_data = None
        self.test_data = None
        self.full_dataset = None
        
    def load_data(self):
        """Load the synthetic training and test datasets"""
        pkl_path = os.path.join(self.data_root, 'train_test_split.pkl')
        
        print(f"Loading data from: {pkl_path}")
        
        try:
            with open(pkl_path, 'rb') as f:
                data_split = dill.load(f)
            
            print(f"Loaded data type: {type(data_split)}")
            print(f"Data structure: {data_split}")
            
            # Handle different data structures
            if hasattr(data_split, 'train_dataset'):
                # Object with attributes
                self.train_data = data_split.train_dataset
                self.test_data = data_split.test_dataset
            elif isinstance(data_split, (tuple, list)) and len(data_split) >= 2:
                # Tuple/list format: (train_data, test_data)
                self.train_data = data_split[0]
                self.test_data = data_split[1]
                print(f"Extracted from tuple/list format")
            elif isinstance(data_split, dict):
                # Dictionary format
                if 'train_dataset' in data_split:
                    self.train_data = data_split['train_dataset']
                    self.test_data = data_split['test_dataset']
                elif 'train' in data_split:
                    self.train_data = data_split['train']
                    self.test_data = data_split['test']
                else:
                    # Print available keys to help debug
                    print(f"Dictionary keys: {list(data_split.keys())}")
                    raise ValueError("Could not find train/test data in dictionary")
            else:
                print(f"Unexpected data format. Type: {type(data_split)}")
                if hasattr(data_split, '__dict__'):
                    print(f"Object attributes: {list(data_split.__dict__.keys())}")
                raise ValueError("Unknown data format")
            
            print(f"Successfully loaded data:")
            print(f"  Train dataset type: {type(self.train_data)}")
            print(f"  Test dataset type: {type(self.test_data)}")
            
            # Try to get shape information
            if hasattr(self.train_data, '__len__'):
                print(f"  Train dataset length: {len(self.train_data)}")
            if hasattr(self.test_data, '__len__'):
                print(f"  Test dataset length: {len(self.test_data)}")
                
            # If it's a tuple/list, let's also inspect the first few elements
            if isinstance(data_split, (tuple, list)):
                print(f"  Total elements in loaded structure: {len(data_split)}")
                for i, elem in enumerate(data_split[:3]):  # Show first 3 elements
                    print(f"  Element {i}: type={type(elem)}, shape={getattr(elem, 'shape', 'no shape attr')}")
                
        except Exception as e:
            print(f"Error loading data: {e}")
            raise
    
    def extract_data_points(self, dataset, max_batches=None):
        """Extract data points from dataset (assuming it's a DataLoader or similar)"""
        rows = []
        batch_count = 0
        
        try:
            # If dataset is iterable (like DataLoader)
            for batch_data in dataset:
                if isinstance(batch_data, (list, tuple)):
                    xb = batch_data[0]  # First element should be the data
                else:
                    xb = batch_data
                
                # Convert to numpy if it's a tensor
                if hasattr(xb, 'detach'):
                    x_np = xb.detach().cpu().numpy()
                else:
                    x_np = np.array(xb)
                
                # Reshape to 2D if needed
                if x_np.ndim > 2:
                    x_np = x_np.reshape(x_np.shape[0], -1)
                elif x_np.ndim == 1:
                    x_np = x_np.reshape(1, -1)
                
                rows.append(x_np)
                batch_count += 1
                
                if max_batches and batch_count >= max_batches:
                    break
                    
        except Exception as e:
            print(f"Error extracting data points: {e}")
            # If dataset is already a numpy array or similar
            if hasattr(dataset, 'shape'):
                return dataset.reshape(dataset.shape[0], -1)
            raise
        
        if rows:
            data_points = np.vstack(rows)
            print(f"Extracted {data_points.shape[0]} data points with {data_points.shape[1]} features")
            return data_points
        else:
            raise ValueError("No data points could be extracted from dataset")
    
    def find_k_for_subset(self, subset, max_k=100):
        """Find optimal k for a given subset"""
        k = 1
        while k < max_k:
            try:
                # Can't have more neighbors than data points
                if k >= len(subset):
                    return None
                
                # Create k-NN graph
                neighbors = NearestNeighbors(n_neighbors=k+1).fit(subset)  # +1 because includes self
                graph = neighbors.kneighbors_graph(subset, mode='connectivity')
                
                # Remove self-connections and make undirected
                graph.setdiag(0)
                graph_undirected = graph.maximum(graph.T)
                graph_undirected.eliminate_zeros()
                
                # Count connected components
                n_components, _ = connected_components(csgraph=graph_undirected, directed=False)
                
                if n_components == self.true_b0:
                    return k  # Found the k that gives us the right B0
                k += 1
                
            except Exception as e:
                k += 1
                continue
        
        return None  # No suitable k found
    
    def monte_carlo_k_optimization(self, n_trials=1000, subset_fraction=0.25, max_k=100, use_train_data=True):
        """Run Monte Carlo simulation to find optimal k using 25% of data each time"""
        print(f"\nRunning Monte Carlo optimization:")
        print(f"  Number of trials: {n_trials}")
        print(f"  Using {subset_fraction*100}% of data each trial")
        print(f"  Target B0: {self.true_b0}")
        print(f"  Max k to test: {max_k}")
        
        # Choose dataset and extract all points
        dataset = self.train_data if use_train_data else self.test_data
        self.full_dataset = self.extract_data_points(dataset)
        
        # Calculate subset size as 25% of total data
        total_points = len(self.full_dataset)
        subset_size = int(total_points * subset_fraction)
        
        print(f"  Total data points: {total_points}")
        print(f"  Subset size (25%): {subset_size}")
        
        if subset_size < 50:  # Minimum reasonable subset size
            print(f"Warning: Subset size {subset_size} is very small. Consider using more data.")
        
        successful_ks = []
        failed_trials = 0
        
        # Use tqdm for progress bar in Jupyter
        for trial in tqdm(range(n_trials), desc="Monte Carlo trials"):
            try:
                # Generate random subset (25% of data)
                subset_indices = random.sample(range(total_points), subset_size)
                subset = self.full_dataset[subset_indices]
                
                # Find optimal k for this subset
                k = self.find_k_for_subset(subset, max_k)
                
                if k is not None:
                    successful_ks.append(k)
                else:
                    failed_trials += 1
                    
            except Exception as e:
                failed_trials += 1
                continue
        
        print(f"\nMonte Carlo Results:")
        print(f"  Successful trials: {len(successful_ks)}")
        print(f"  Failed trials: {failed_trials}")
        print(f"  Success rate: {len(successful_ks)/n_trials*100:.1f}%")
        
        return successful_ks
    
    def analyze_k_distribution(self, successful_ks):
        """Analyze the distribution of successful k values"""
        if not successful_ks:
            print("No successful k values to analyze!")
            return None, None
        
        k_array = np.array(successful_ks)
        mean_k = np.mean(k_array)
        std_k = np.std(k_array)
        median_k = np.median(k_array)
        mode_result = Counter(successful_ks).most_common(1)
        mode_k = mode_result[0][0] if mode_result else None
        
        print(f"\nK Distribution Analysis:")
        print(f"  Mean k: {mean_k:.2f}")
        print(f"  Median k: {median_k:.2f}")
        print(f"  Mode k: {mode_k}")
        print(f"  Standard deviation: {std_k:.2f}")
        print(f"  Min k: {min(successful_ks)}")
        print(f"  Max k: {max(successful_ks)}")
        
        # Create histogram
        plt.figure(figsize=(15, 5))
        
        plt.subplot(1, 3, 1)
        plt.hist(successful_ks, bins=max(20, len(set(successful_ks))), alpha=0.7, edgecolor='black')
        plt.axvline(mean_k, color='red', linestyle='--', label=f'Mean: {mean_k:.2f}', linewidth=2)
        plt.axvline(median_k, color='green', linestyle='--', label=f'Median: {median_k:.2f}', linewidth=2)
        if mode_k:
            plt.axvline(mode_k, color='orange', linestyle='--', label=f'Mode: {mode_k}', linewidth=2)
        plt.xlabel('k value')
        plt.ylabel('Frequency')
        plt.title('Distribution of Optimal k Values')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 3, 2)
        k_counts = Counter(successful_ks)
        ks, counts = zip(*sorted(k_counts.items()))
        plt.bar(ks, counts, alpha=0.7)
        plt.xlabel('k value')
        plt.ylabel('Count')
        plt.title('Count of Each k Value')
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 3, 3)
        plt.boxplot(successful_ks)
        plt.ylabel('k value')
        plt.title('Box Plot of k Values')
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        return mean_k, std_k
    
    def find_best_subset_for_k(self, target_k, n_attempts=100, subset_fraction=0.25):
        """Find a subset that works well with the target k value"""
        print(f"\nFinding best subset for k={target_k}:")
        
        if self.full_dataset is None:
            raise ValueError("Must run monte_carlo_k_optimization first to load full dataset")
        
        total_points = len(self.full_dataset)
        subset_size = int(total_points * subset_fraction)
        target_k_int = int(round(target_k))
        
        print(f"  Looking for subset of size {subset_size} (25% of {total_points} points)")
        
        best_subset = None
        
        for attempt in tqdm(range(n_attempts), desc="Finding best subset"):
            try:
                subset_indices = random.sample(range(total_points), subset_size)
                subset = self.full_dataset[subset_indices]
                
                # Test if this subset gives the right B0 with our target k
                found_k = self.find_k_for_subset(subset, max_k=target_k_int+5)
                
                if found_k == target_k_int:
                    best_subset = subset
                    print(f"  ✅ Found suitable subset on attempt {attempt+1}")
                    break
                    
            except Exception as e:
                continue
        
        if best_subset is None:
            print(f"  ❌ Could not find subset that works with k={target_k_int} after {n_attempts} attempts")
            print(f"     Try increasing n_attempts or adjusting target_k")
        
        return best_subset

In [ ]:
# initialize and load the data
# Initialize the optimizer
optimizer = MonteCarloKNNOptimizer(
    data_root='${TDL_ROOT_DIR}/results/D1',
    true_b0=9
)

# Load the data
optimizer.load_data()

In [ ]:
# Run a quick test with fewer trials to make sure everything works
print("=== Quick Test (100 trials) ===")

successful_ks_test = optimizer.monte_carlo_k_optimization(
    n_trials=100,           # Small number for testing
    subset_fraction=0.25,   # Use 25% of data each time
    max_k=30,              # Don't test very high k values
    use_train_data=True    # Use training data
)

if successful_ks_test:
    print(f"\n✅ Quick test successful! Found {len(successful_ks_test)} valid k values")
    print(f"Sample k values: {successful_ks_test[:9]}")
else:
    print("❌ Quick test failed - check data loading")

# Analyze the quick test results
if successful_ks_test:
    mean_k_test, std_k_test = optimizer.analyze_k_distribution(successful_ks_test)
    print(f"Quick test mean k: {mean_k_test:.2f}")

In [ ]:
# Run the full Monte Carlo optimization
print("=== Full Monte Carlo Optimization ===")

successful_ks = optimizer.monte_carlo_k_optimization(
    n_trials=1000,          # Full number of trials
    subset_fraction=0.25,   # Use 25% of data each time
    max_k=50,              # Test up to k=50
    use_train_data=True
)

print(f"\nFound {len(successful_ks)} successful k values out of 1000 trials")

In [ ]:
# Analyze the full results
if successful_ks:
    mean_k, std_k = optimizer.analyze_k_distribution(successful_ks)
    
    # Print summary statistics
    print(f"\n📊 SUMMARY RESULTS:")
    print(f"   Optimal k (mean): {mean_k:.2f}")
    print(f"   Standard deviation: {std_k:.2f}")
    print(f"   Success rate: {len(successful_ks)/1000*100:.1f}%")
else:
    print("❌ No successful k values found")

In [ ]:
# Find a specific subset that works with the mean k
if 'mean_k' in locals() and mean_k:
    print(f"Finding subset that works with k = {int(round(mean_k))}")
    
    best_subset = optimizer.find_best_subset_for_k(
        target_k=mean_k,
        n_attempts=100,
        subset_fraction=0.25
    )
    
    if best_subset is not None:
        print(f"\n🎯 FINAL RESULTS:")
        print(f"   Optimal k: {int(round(mean_k))}")
        print(f"   Subset size: {len(best_subset)}")
        print(f"   Data shape: {best_subset.shape}")
        print(f"   Ready for outlier removal and Ripser analysis!")
        
        # Verify it works
        k_verify = optimizer.find_k_for_subset(best_subset, max_k=int(round(mean_k))+1)
        print(f"   Verification: k={k_verify} gives B0={optimizer.true_b0} ✅")
    else:
        print("❌ Could not find suitable subset")

In [ ]:
# Save all results for later use
if 'best_subset' in locals() and best_subset is not None:
    results = {
        'optimal_k': int(round(mean_k)),
        'mean_k': mean_k,
        'std_k': std_k,
        'all_successful_ks': successful_ks,
        'best_subset': best_subset,
        'subset_shape': best_subset.shape,
        'data_fraction_used': 0.25,
        'true_b0': optimizer.true_b0
    }
    
    # Save to pickle file
    with open('monte_carlo_results.pkl', 'wb') as f:
        pickle.dump(results, f)
    
    print("💾 Results saved to: monte_carlo_results.pkl")
    print("\nNext steps:")
    print("1. Remove outliers from best_subset")
    print("2. Run Ripser on cleaned data") 
    print("3. Track components through layers")
    
    # Also save just the subset as numpy array for easy loading
    np.save('optimal_subset.npy', best_subset)
    print("📁 Subset saved to: optimal_subset.npy")

In [ ]:
# Inspect the optimal subset
if 'best_subset' in locals() and best_subset is not None:
    plt.figure(figsize=(10, 6))
    
    # If 2D data, plot scatter
    if best_subset.shape[1] == 2:
        plt.subplot(1, 2, 1)
        plt.scatter(best_subset[:, 0], best_subset[:, 1], alpha=0.6, s=20)
        plt.title(f'Optimal Subset (k={int(round(mean_k))})')
        plt.xlabel('Feature 1')
        plt.ylabel('Feature 2')
        plt.grid(True, alpha=0.3)
    
    # Plot distribution of features
    plt.subplot(1, 2, 2)
    for i in range(min(best_subset.shape[1], 3)):  # Plot up to 3 features
        plt.hist(best_subset[:, i], alpha=0.5, label=f'Feature {i+1}', bins=20)
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.title('Feature Distributions')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Subset statistics:")
    print(f"  Shape: {best_subset.shape}")
    print(f"  Mean: {np.mean(best_subset, axis=0)}")
    print(f"  Std: {np.std(best_subset, axis=0)}")

In [ ]:
# Left plot: Several distinct groups of points
# Right plot: Multiple peaks in histograms
# → This explains why B0=9 (9 connected components)